# Landslide step 03: post-processing and summaries (minimum_scenario)

Builds grouped damage layers and summary tables for this scenario set, equivalent to the coastal step-03 workflow.

In [ ]:
import os
import re
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd


In [ ]:
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
output_path = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages/results_landslide_minimum_scenario'

data_root = base_path / 'dphil_common_cross_cutting/common_incoming_data'
network_csv = data_root / 'networks/network_layers_hazard_intersections_details.csv'

jamaica_crs = 3448

print('Output path:', output_path)
print('Network csv:', network_csv)


In [ ]:
damage_results_folder = output_path / 'direct_damages'
damage_estimates_directory = output_path / 'damage_estimates'
damage_estimates_directory.mkdir(parents=True, exist_ok=True)

hazard_column_pattern = re.compile(r'^landslide_(baseline|deforestation|reforestation)_rp_(\d+)$')


def resolve_network_asset_file(asset_relative_path):
    relative_asset_path = Path(asset_relative_path)
    asset_file_in_common_incoming_data = data_root / relative_asset_path
    asset_file_in_nested_networks_folder = data_root / 'networks' / relative_asset_path

    if asset_file_in_common_incoming_data.exists():
        return asset_file_in_common_incoming_data
    if asset_file_in_nested_networks_folder.exists():
        return asset_file_in_nested_networks_folder

    raise FileNotFoundError(
        f"Could not find asset file '{relative_asset_path}'. Checked: {asset_file_in_common_incoming_data} ; {asset_file_in_nested_networks_folder}"
    )


def get_landslide_damage_columns(columns):
    return [column_name for column_name in columns if hazard_column_pattern.match(column_name)]


In [ ]:
asset_data_details = pd.read_csv(network_csv)
damage_totals = []
missing_damage_files = []

for asset_info in asset_data_details.itertuples(index=False):
    asset_gpkg = asset_info.asset_gpkg
    asset_layer = asset_info.asset_layer
    asset_id = asset_info.asset_id_column

    df_path = damage_results_folder / f'{asset_gpkg}_{asset_layer}' / f'{asset_gpkg}_{asset_layer}_direct_damages_parameter_set_0.parquet'

    if not df_path.exists():
        missing_damage_files.append(str(df_path))
        continue

    df = pd.read_parquet(df_path)
    landslide_damage_columns = get_landslide_damage_columns(df.columns)

    if len(landslide_damage_columns) == 0:
        print(f'No landslide damage columns found in {df_path.name}, skipping')
        continue

    damage_uncertainty_parameter = (
        df['damage_uncertainty_parameter'].iloc[0]
        if ('damage_uncertainty_parameter' in df.columns and not df.empty)
        else np.nan
    )
    cost_uncertainty_parameter = (
        df['cost_uncertainty_parameter'].iloc[0]
        if ('cost_uncertainty_parameter' in df.columns and not df.empty)
        else np.nan
    )

    # Asset-level damages
    df_grouped = df.groupby([asset_id], as_index=False)[landslide_damage_columns].sum()

    resolved_asset_file = resolve_network_asset_file(asset_info.path)
    df_geom = gpd.read_file(resolved_asset_file, layer=asset_layer)
    df_geom = df_geom.to_crs(epsg=jamaica_crs)

    df_grouped = pd.merge(df_grouped, df_geom[[asset_id, 'geometry']], how='left', on=[asset_id])
    df_grouped = gpd.GeoDataFrame(df_grouped, geometry='geometry', crs=jamaica_crs)

    output_geopackage = damage_estimates_directory / f'{asset_gpkg}_{asset_layer}_asset_damages_groupedby.gpkg'
    df_grouped.to_file(output_geopackage, driver='GPKG')

    # Sector/layer totals
    df_totals = df_grouped[landslide_damage_columns].sum().to_frame().T
    df_totals['sector'] = asset_gpkg
    df_totals['layer'] = asset_layer
    df_totals['damage_uncertainty_parameter'] = damage_uncertainty_parameter
    df_totals['cost_uncertainty_parameter'] = cost_uncertainty_parameter
    damage_totals.append(df_totals)

if damage_totals:
    damage_totals = pd.concat(damage_totals, axis=0, ignore_index=True)
else:
    damage_totals = pd.DataFrame(columns=['sector', 'layer', 'damage_uncertainty_parameter', 'cost_uncertainty_parameter'])

damage_totals.to_csv(damage_estimates_directory / 'asset_damages_groupedby.csv', index=False)

print('Grouped outputs created:', len(damage_totals))
if missing_damage_files:
    print('Missing direct-damage files:', len(missing_damage_files))


In [ ]:
if not damage_estimates_directory.exists():
    raise FileNotFoundError(f'Missing folder: {damage_estimates_directory}')

damage_estimate_files = sorted(damage_estimates_directory.glob('*_asset_damages_groupedby.gpkg'))
if not damage_estimate_files:
    raise FileNotFoundError(f'No grouped damage GPKGs found in {damage_estimates_directory}')

summary_rows = []
for grouped_damage_file in damage_estimate_files:
    grouped_damage = gpd.read_file(grouped_damage_file)
    asset_name = grouped_damage_file.stem.replace('_asset_damages_groupedby', '')
    summary_rows.append({
        'asset_name': asset_name,
        'row_count': len(grouped_damage),
        'column_count': len(grouped_damage.columns),
    })

pd.DataFrame(summary_rows).sort_values('asset_name').reset_index(drop=True)


In [ ]:
# Choose one output to inspect
asset_name_to_preview = 'roads_edges'  # e.g. rail_nodes, roads_edges, buildings_assigned_economic_activity_areas
preview_file = damage_estimates_directory / f'{asset_name_to_preview}_asset_damages_groupedby.gpkg'

if not preview_file.exists():
    raise FileNotFoundError(f'Missing file: {preview_file}')

preview_table = gpd.read_file(preview_file)
print(f'Rows: {len(preview_table)} | Columns: {len(preview_table.columns)}')
preview_table.head(20)


In [ ]:
# Sum damages by Sector, Subsector, Scenario, and ReturnPeriod.
# Output in both JMD/JD and USD using strict conversion: 150 JMD = 1 USD.
network_details = pd.read_csv(network_csv)[['sector', 'asset_description', 'asset_gpkg', 'asset_layer']].drop_duplicates()

JMD_PER_USD = 150.0

damage_rows = []
for asset_info in network_details.itertuples(index=False):
    damage_file = (
        output_path
        / 'direct_damages'
        / f'{asset_info.asset_gpkg}_{asset_info.asset_layer}'
        / f'{asset_info.asset_gpkg}_{asset_info.asset_layer}_direct_damages_parameter_set_0.parquet'
    )

    if not damage_file.exists():
        continue

    damage_table = pd.read_parquet(damage_file)
    landslide_damage_columns = get_landslide_damage_columns(damage_table.columns)

    for column_name in landslide_damage_columns:
        match = hazard_column_pattern.match(column_name)
        scenario = match.group(1)
        return_period = int(match.group(2))
        direct_damages_jd = float(damage_table[column_name].sum())
        direct_damages_usd = direct_damages_jd / JMD_PER_USD

        damage_rows.append({
            'Sector': asset_info.sector,
            'Subsector': asset_info.asset_description,
            'Asset': asset_info.asset_gpkg,
            'Layer': asset_info.asset_layer,
            'Scenario': scenario,
            'ReturnPeriod': return_period,
            'Direct_Damages_JD': direct_damages_jd,
            'Direct_Damages_USD': direct_damages_usd,
        })

asset_level_summary = pd.DataFrame(damage_rows)

if asset_level_summary.empty:
    raise ValueError('No landslide direct-damage summaries were produced. Run step 02 first.')

sector_subsector_summary = (
    asset_level_summary
    .groupby(['Sector', 'Subsector', 'Scenario', 'ReturnPeriod'], as_index=False)[['Direct_Damages_JD', 'Direct_Damages_USD']]
    .sum()
    .sort_values(['Sector', 'Subsector', 'Scenario', 'ReturnPeriod'])
)

sector_summary = (
    sector_subsector_summary
    .groupby(['Sector', 'Scenario', 'ReturnPeriod'], as_index=False)[['Direct_Damages_JD', 'Direct_Damages_USD']]
    .sum()
    .sort_values(['Sector', 'Scenario', 'ReturnPeriod'])
)

# Scenario deltas vs baseline (both JD and USD)
pivot = (
    sector_subsector_summary
    .pivot_table(index=['Sector', 'Subsector', 'ReturnPeriod'], columns='Scenario', values=['Direct_Damages_JD', 'Direct_Damages_USD'], aggfunc='sum', fill_value=0.0)
)

for unit in ['JD', 'USD']:
    for scenario in ['baseline', 'deforestation', 'reforestation']:
        if (f'Direct_Damages_{unit}', scenario) not in pivot.columns:
            pivot[(f'Direct_Damages_{unit}', scenario)] = 0.0

pivot = pivot.reset_index()

pivot['Deforestation_Change_JD'] = pivot[('Direct_Damages_JD', 'deforestation')] - pivot[('Direct_Damages_JD', 'baseline')]
pivot['Reforestation_Change_JD'] = pivot[('Direct_Damages_JD', 'reforestation')] - pivot[('Direct_Damages_JD', 'baseline')]
pivot['Deforestation_Change_USD'] = pivot[('Direct_Damages_USD', 'deforestation')] - pivot[('Direct_Damages_USD', 'baseline')]
pivot['Reforestation_Change_USD'] = pivot[('Direct_Damages_USD', 'reforestation')] - pivot[('Direct_Damages_USD', 'baseline')]

# Flatten multi-index columns after pivot
pivot.columns = [
    col if isinstance(col, str) else '_'.join([str(part) for part in col if str(part) != '']).strip('_')
    for col in pivot.columns
]

sector_subsector_summary_file = output_path / 'damage_estimates' / 'sector_subsector_scenario_return_period_damages.csv'
sector_summary_file = output_path / 'damage_estimates' / 'sector_scenario_return_period_damages.csv'
delta_summary_file = output_path / 'damage_estimates' / 'scenario_deltas_vs_baseline.csv'

sector_subsector_summary.to_csv(sector_subsector_summary_file, index=False)
sector_summary.to_csv(sector_summary_file, index=False)
pivot.to_csv(delta_summary_file, index=False)

print(f'Saved: {sector_subsector_summary_file}')
print(f'Saved: {sector_summary_file}')
print(f'Saved: {delta_summary_file}')

print()
print('Sector + Subsector summary:')
display(sector_subsector_summary)
print()
print('Sector-only summary:')
display(sector_summary)
print()
print('Scenario deltas vs baseline:')
display(pivot)


In [ ]:
# RP analysis: damages and avoided damages across sectors
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

scenario_order = ['baseline', 'deforestation', 'reforestation']
sector_order = ['buildings', 'transport', 'water', 'energy']

sector_summary_usd = sector_summary.copy()
sector_summary_usd['ReturnPeriod'] = pd.to_numeric(sector_summary_usd['ReturnPeriod'], errors='coerce').astype(int)
sector_summary_usd['Direct_Damages_USD'] = pd.to_numeric(sector_summary_usd['Direct_Damages_USD'], errors='coerce').fillna(0.0)

# Build a wide table for scenario comparison by sector and RP
sector_rp_scenario = (
    sector_summary_usd
    .pivot_table(
        index=['Sector', 'ReturnPeriod'],
        columns='Scenario',
        values='Direct_Damages_USD',
        aggfunc='sum',
        fill_value=0.0,
    )
    .reset_index()
)

for c in scenario_order:
    if c not in sector_rp_scenario.columns:
        sector_rp_scenario[c] = 0.0

sector_rp_scenario['Avoided_Protection_USD'] = sector_rp_scenario['deforestation'] - sector_rp_scenario['baseline']
sector_rp_scenario['Avoided_Restoration_USD'] = sector_rp_scenario['baseline'] - sector_rp_scenario['reforestation']
sector_rp_scenario['Combined_Benefit_USD'] = sector_rp_scenario['deforestation'] - sector_rp_scenario['reforestation']

sector_rp_scenario = sector_rp_scenario.sort_values(['Sector', 'ReturnPeriod']).reset_index(drop=True)

# Save tables
sector_rp_scenario_file = damage_estimates_directory / 'sector_return_period_scenario_damages_usd.csv'
sector_rp_avoided_file = damage_estimates_directory / 'sector_return_period_avoided_damages_usd.csv'

sector_rp_scenario.to_csv(sector_rp_scenario_file, index=False)
sector_rp_scenario[[
    'Sector', 'ReturnPeriod',
    'Avoided_Protection_USD', 'Avoided_Restoration_USD', 'Combined_Benefit_USD'
]].to_csv(sector_rp_avoided_file, index=False)

print('Saved:', sector_rp_scenario_file)
print('Saved:', sector_rp_avoided_file)

# Plot 1: direct damages vs RP across sectors (three scenarios)
plot_sectors = [s for s in sector_order if s in sector_rp_scenario['Sector'].unique()]
plot_sectors += sorted([s for s in sector_rp_scenario['Sector'].unique() if s not in plot_sectors])

rps = sorted(sector_rp_scenario['ReturnPeriod'].unique().tolist())

n = len(plot_sectors)
ncols = 2
nrows = int(np.ceil(n / ncols))

scenario_styles = {
    'baseline': '#4C78A8',
    'deforestation': '#F58518',
    'reforestation': '#54A24B',
}

fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.1 * nrows), sharex=True)
axes = np.array(axes).reshape(-1)

for i, sector_name in enumerate(plot_sectors):
    ax = axes[i]
    d = sector_rp_scenario[sector_rp_scenario['Sector'] == sector_name].sort_values('ReturnPeriod')

    for sc in scenario_order:
        ax.plot(
            d['ReturnPeriod'],
            d[sc],
            marker='o',
            linewidth=2,
            markersize=4,
            color=scenario_styles[sc],
            label=sc.capitalize(),
        )

    ax.set_title(sector_name.capitalize())
    ax.set_xticks(rps)
    ax.set_xlabel('Return period (years)')
    ax.set_ylabel('Direct damages (USD billion)')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f'${v/1e9:,.2f}bn'))
    ax.grid(alpha=0.25)

for j in range(n, len(axes)):
    axes[j].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, frameon=True, bbox_to_anchor=(0.5, 1.02))
fig.suptitle('Landslide Direct Damages by Return Period Across Sectors', y=1.05)
plt.tight_layout()

direct_rp_chart_file = damage_estimates_directory / 'landslide_sector_direct_damages_by_rp_and_scenario.png'
fig.savefig(direct_rp_chart_file, dpi=300, bbox_inches='tight')
print('Saved:', direct_rp_chart_file)
plt.show()

# Plot 2: avoided damages vs RP across sectors
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.1 * nrows), sharex=True)
axes = np.array(axes).reshape(-1)

for i, sector_name in enumerate(plot_sectors):
    ax = axes[i]
    d = sector_rp_scenario[sector_rp_scenario['Sector'] == sector_name].sort_values('ReturnPeriod')

    ax.plot(
        d['ReturnPeriod'],
        d['Avoided_Restoration_USD'],
        marker='o',
        linewidth=2,
        markersize=4,
        color='#2E7D32',
        label='Avoided restoration (Baseline - Reforestation)',
    )
    ax.plot(
        d['ReturnPeriod'],
        d['Avoided_Protection_USD'],
        marker='o',
        linewidth=2,
        markersize=4,
        color='#EF6C00',
        label='Avoided protection (Deforestation - Baseline)',
    )

    ax.axhline(0.0, color='#777777', linewidth=1, linestyle='--')
    ax.set_title(sector_name.capitalize())
    ax.set_xticks(rps)
    ax.set_xlabel('Return period (years)')
    ax.set_ylabel('Avoided damages (USD billion)')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f'${v/1e9:,.2f}bn'))
    ax.grid(alpha=0.25)

for j in range(n, len(axes)):
    axes[j].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=1, frameon=True, bbox_to_anchor=(0.5, 1.06))
fig.suptitle('Landslide Avoided Damages by Return Period Across Sectors', y=1.10)
plt.tight_layout()

avoided_rp_chart_file = damage_estimates_directory / 'landslide_sector_avoided_damages_by_rp.png'
fig.savefig(avoided_rp_chart_file, dpi=300, bbox_inches='tight')
print('Saved:', avoided_rp_chart_file)
plt.show()

display(sector_rp_scenario.head(30))

In [ ]:
# RP bar charts: total and sector direct damages
scenario_order = ['baseline', 'deforestation', 'reforestation']
scenario_colors = {
    'baseline': '#4C78A8',
    'deforestation': '#F58518',
    'reforestation': '#54A24B',
}

# Total (all sectors) direct damages by return period
rp_total = (
    sector_rp_scenario
    .groupby('ReturnPeriod', as_index=False)[scenario_order]
    .sum()
    .sort_values('ReturnPeriod')
)

bar_x = np.arange(len(rp_total))
bar_w = 0.24

fig, ax = plt.subplots(figsize=(10.5, 6))
for i, sc in enumerate(scenario_order):
    ax.bar(
        bar_x + (i - 1) * bar_w,
        rp_total[sc].to_numpy(),
        width=bar_w,
        label=sc.capitalize(),
        color=scenario_colors[sc],
        edgecolor='white',
        linewidth=0.8,
    )

ax.set_xticks(bar_x)
ax.set_xticklabels([str(int(v)) for v in rp_total['ReturnPeriod']])
ax.set_xlabel('Return period (years)')
ax.set_ylabel('Total direct damages (USD billion)')
ax.set_title('Total Landslide Direct Damages by Return Period')
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f'${v/1e9:,.2f}bn'))
ax.grid(axis='y', alpha=0.25)
ax.legend(title='Scenario', frameon=True)
plt.tight_layout()

total_bar_file = damage_estimates_directory / 'landslide_total_direct_damages_by_rp_bar.png'
fig.savefig(total_bar_file, dpi=300, bbox_inches='tight')
print('Saved:', total_bar_file)
plt.show()

# Bar chart for each sector (faceted), direct damages by RP and scenario
sector_order = ['buildings', 'transport', 'water', 'energy']
plot_sectors = [s for s in sector_order if s in sector_rp_scenario['Sector'].unique()]
plot_sectors += sorted([s for s in sector_rp_scenario['Sector'].unique() if s not in plot_sectors])

n = len(plot_sectors)
ncols = 2
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.2 * nrows), sharex=True)
axes = np.array(axes).reshape(-1)

for idx, sector_name in enumerate(plot_sectors):
    ax = axes[idx]
    d = sector_rp_scenario[sector_rp_scenario['Sector'] == sector_name].sort_values('ReturnPeriod')
    x = np.arange(len(d))

    for i, sc in enumerate(scenario_order):
        ax.bar(
            x + (i - 1) * bar_w,
            d[sc].to_numpy(),
            width=bar_w,
            color=scenario_colors[sc],
            edgecolor='white',
            linewidth=0.7,
            label=sc.capitalize(),
        )

    ax.set_title(sector_name.capitalize())
    ax.set_xticks(x)
    ax.set_xticklabels([str(int(v)) for v in d['ReturnPeriod']])
    ax.set_xlabel('Return period (years)')
    ax.set_ylabel('Direct damages (USD billion)')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f'${v/1e9:,.2f}bn'))
    ax.grid(axis='y', alpha=0.25)

for j in range(n, len(axes)):
    axes[j].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, frameon=True, bbox_to_anchor=(0.5, 1.02))
fig.suptitle('Sector Direct Damages by Return Period', y=1.05)
plt.tight_layout()

sector_bar_file = damage_estimates_directory / 'landslide_sector_direct_damages_by_rp_bar_panels.png'
fig.savefig(sector_bar_file, dpi=300, bbox_inches='tight')
print('Saved:', sector_bar_file)
plt.show()

rp_total